In [ ]:
#!/usr/bin/env python3
"""Pose estimation + workout counter from USB webcam -> DisplayPort.

Counts squats, pushups, and situps using MoveNet Lightning keypoints.
Push buttons on the PL:
  BTN0: clear current user's counts
  BTN1: next user
  BTN2: previous user
  BTN3: reset all users
"""
import cv2
import numpy as np
import time
import tflite_runtime.interpreter as tflite
from pynq import Overlay
from pynq.lib.video import DisplayPort, VideoMode, PIXEL_RGB

MODEL_PATH = "movenet_lightning.tflite"
BITSTREAM = "workout_classifier.bit"
DISPLAY_W, DISPLAY_H = 1920, 1080
CAM_W, CAM_H = 640, 480
CONF_THRESH = 0.25
NUM_USERS = 4

# MoveNet keypoint indices
NOSE = 0
L_SHOULDER, R_SHOULDER = 5, 6
L_ELBOW, R_ELBOW = 7, 8
L_WRIST, R_WRIST = 9, 10
L_HIP, R_HIP = 11, 12
L_KNEE, R_KNEE = 13, 14
L_ANKLE, R_ANKLE = 15, 16

SKELETON = [
    (0, 1), (0, 2), (1, 3), (2, 4),
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]


def angle_between(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba, bc = a - b, c - b
    cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0)))


def kp_xy(kps, idx, w, h):
    y, x, _ = kps[idx]
    return (x * w, y * h)


def kp_ok(kps, *idx):
    return all(kps[i][2] >= CONF_THRESH for i in idx)


class ExerciseCounter:
    def __init__(self, name, color):
        self.name = name
        self.color = color
        self.count = 0
        self.state = "up"
        self.angle = 0.0
        self.active = False

    def reset(self):
        self.count = 0
        self.state = "up"


class UserStats:
    def __init__(self, name):
        self.name = name
        self.squat = ExerciseCounter("Squats", (0, 200, 255))
        self.pushup = ExerciseCounter("Pushups", (255, 120, 0))
        self.situp = ExerciseCounter("Situps", (0, 255, 120))

    def reset(self):
        self.squat.reset()
        self.pushup.reset()
        self.situp.reset()

    def update(self, kps, w, h):
        self._squat(kps, w, h)
        self._pushup(kps, w, h)
        self._situp(kps, w, h)

    # Squat: knee angle, standing ~170, bottom ~90
    def _squat(self, kps, w, h):
        if not kp_ok(kps, L_HIP, L_KNEE, L_ANKLE, R_HIP, R_KNEE, R_ANKLE):
            self.squat.active = False
            return
        self.squat.active = True
        l = angle_between(kp_xy(kps, L_HIP, w, h),
                          kp_xy(kps, L_KNEE, w, h),
                          kp_xy(kps, L_ANKLE, w, h))
        r = angle_between(kp_xy(kps, R_HIP, w, h),
                          kp_xy(kps, R_KNEE, w, h),
                          kp_xy(kps, R_ANKLE, w, h))
        ang = (l + r) / 2
        self.squat.angle = ang
        if ang < 100 and self.squat.state == "up":
            self.squat.state = "down"
        elif ang > 160 and self.squat.state == "down":
            self.squat.state = "up"
            self.squat.count += 1

    # Pushup: elbow angle + body roughly horizontal
    def _pushup(self, kps, w, h):
        if not kp_ok(kps, L_SHOULDER, L_ELBOW, L_WRIST,
                    R_SHOULDER, R_ELBOW, R_WRIST, L_HIP, R_HIP):
            self.pushup.active = False
            return
        sh_y = (kps[L_SHOULDER][0] + kps[R_SHOULDER][0]) / 2
        hi_y = (kps[L_HIP][0] + kps[R_HIP][0]) / 2
        if abs(sh_y - hi_y) * h > 120:  # too vertical — not pushup stance
            self.pushup.active = False
            return
        self.pushup.active = True
        l = angle_between(kp_xy(kps, L_SHOULDER, w, h),
                          kp_xy(kps, L_ELBOW, w, h),
                          kp_xy(kps, L_WRIST, w, h))
        r = angle_between(kp_xy(kps, R_SHOULDER, w, h),
                          kp_xy(kps, R_ELBOW, w, h),
                          kp_xy(kps, R_WRIST, w, h))
        ang = (l + r) / 2
        self.pushup.angle = ang
        if ang < 90 and self.pushup.state == "up":
            self.pushup.state = "down"
        elif ang > 150 and self.pushup.state == "down":
            self.pushup.state = "up"
            self.pushup.count += 1

    # Situp: hip angle (shoulder-hip-knee). Lying flat ~170, sitting up ~70.
    # Requires body roughly horizontal (knees bent / lying down).
    def _situp(self, kps, w, h):
        if not kp_ok(kps, L_SHOULDER, L_HIP, L_KNEE,
                    R_SHOULDER, R_HIP, R_KNEE):
            self.situp.active = False
            return
        # Body should be more horizontal than vertical (person lying down).
        sh_y = (kps[L_SHOULDER][0] + kps[R_SHOULDER][0]) / 2
        hi_y = (kps[L_HIP][0] + kps[R_HIP][0]) / 2
        kn_y = (kps[L_KNEE][0] + kps[R_KNEE][0]) / 2
        # Knees and hips should be around the same vertical band; reject standing.
        if abs(hi_y - kn_y) * h > 160:
            self.situp.active = False
            return
        self.situp.active = True
        l = angle_between(kp_xy(kps, L_SHOULDER, w, h),
                          kp_xy(kps, L_HIP, w, h),
                          kp_xy(kps, L_KNEE, w, h))
        r = angle_between(kp_xy(kps, R_SHOULDER, w, h),
                          kp_xy(kps, R_HIP, w, h),
                          kp_xy(kps, R_KNEE, w, h))
        ang = (l + r) / 2
        self.situp.angle = ang
        if ang < 80 and self.situp.state == "down":
            self.situp.state = "up"
            self.situp.count += 1
        elif ang > 150 and self.situp.state == "up":
            self.situp.state = "down"


def run_movenet(interp, in_det, out_det, size, frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (size, size))
    inp = np.expand_dims(resized, axis=0).astype(np.uint8)
    interp.set_tensor(in_det[0]['index'], inp)
    interp.invoke()
    return interp.get_tensor(out_det[0]['index'])[0, 0]


def draw_pose(frame, kps):
    h, w = frame.shape[:2]
    for a, b in SKELETON:
        ya, xa, ca = kps[a]
        yb, xb, cb = kps[b]
        if ca < CONF_THRESH or cb < CONF_THRESH:
            continue
        cv2.line(frame,
                 (int(xa * w), int(ya * h)),
                 (int(xb * w), int(yb * h)),
                 (0, 255, 120), 2, cv2.LINE_AA)
    for y, x, c in kps:
        if c < CONF_THRESH:
            continue
        cx, cy = int(x * w), int(y * h)
        cv2.circle(frame, (cx, cy), 5, (255, 255, 255), -1, cv2.LINE_AA)
        cv2.circle(frame, (cx, cy), 5, (0, 0, 0), 1, cv2.LINE_AA)


def draw_panel(frame, user, idx, total, fps):
    h, w = frame.shape[:2]
    px = w - 320
    overlay = frame.copy()
    cv2.rectangle(overlay, (px - 10, 0), (w, 360), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)

    cv2.putText(frame, f"User {idx + 1}/{total}: {user.name}",
                (px, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(frame, f"FPS: {fps:4.1f}",
                (px, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1, cv2.LINE_AA)

    for i, ex in enumerate([user.squat, user.pushup, user.situp]):
        y = 95 + i * 85
        dot = ex.color if ex.active else (80, 80, 80)
        cv2.circle(frame, (px + 10, y - 8), 6, dot, -1)
        cv2.putText(frame, ex.name, (px + 28, y - 3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (220, 220, 220), 1, cv2.LINE_AA)
        cv2.putText(frame, str(ex.count), (px + 28, y + 45),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, ex.color, 3, cv2.LINE_AA)
        info = f"{ex.state} {ex.angle:.0f}" if ex.active else "---"
        cv2.putText(frame, info, (px + 140, y + 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (150, 150, 150), 1, cv2.LINE_AA)

    cv2.putText(frame,
                "B0:Clear  B1:Next  B2:Prev  B3:ResetAll",
                (10, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                (200, 200, 200), 1, cv2.LINE_AA)


def fit_to_display(frame_bgr, out_w, out_h):
    h, w = frame_bgr.shape[:2]
    scale = min(out_w / w, out_h / h)
    nw, nh = int(w * scale), int(h * scale)
    scaled = cv2.resize(frame_bgr, (nw, nh))
    canvas = np.zeros((out_h, out_w, 3), dtype=np.uint8)
    x0, y0 = (out_w - nw) // 2, (out_h - nh) // 2
    canvas[y0:y0 + nh, x0:x0 + nw] = scaled
    return canvas


def main():
    interp = tflite.Interpreter(model_path=MODEL_PATH, num_threads=4)
    interp.allocate_tensors()
    in_det = interp.get_input_details()
    out_det = interp.get_output_details()
    size = in_det[0]['shape'][1]
    print(f"MoveNet input: {size}x{size}")

    overlay = Overlay(BITSTREAM)
    btns = overlay.axi_gpio_0.channel1
    btns.setdirection("in")
    btns.setlength(4)
    prev_btn = [0] * 4

    cap = None
    for idx in [0, 1, 2]:
        cap = cv2.VideoCapture(idx, cv2.CAP_V4L2)
        if cap.isOpened():
            print(f"Camera opened at /dev/video{idx}")
            break
        cap.release()
    if not cap or not cap.isOpened():
        raise RuntimeError("No USB webcam found")
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAM_W)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAM_H)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    for _ in range(10):
        cap.read()

    displayport = DisplayPort()
    displayport.configure(VideoMode(DISPLAY_W, DISPLAY_H, 24), PIXEL_RGB)
    print("DisplayPort configured")

    users = [UserStats(f"P{i + 1}") for i in range(NUM_USERS)]
    current = 0

    frame_count = 0
    start = time.time()
    fps = 0.0
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                continue

            # Button edges
            cur = [btns[i].read() for i in range(4)]
            pressed = [c and not p for c, p in zip(cur, prev_btn)]
            prev_btn = cur
            if pressed[0]:
                users[current].reset()
                print(f"Cleared {users[current].name}")
            if pressed[1]:
                current = (current + 1) % NUM_USERS
                print(f"Switched to {users[current].name}")
            if pressed[2]:
                current = (current - 1) % NUM_USERS
                print(f"Switched to {users[current].name}")
            if pressed[3]:
                for u in users:
                    u.reset()
                current = 0
                print("Reset all users")

            h, w = frame.shape[:2]
            kps = run_movenet(interp, in_det, out_det, size, frame)
            users[current].update(kps, w, h)

            draw_pose(frame, kps)
            frame_count += 1
            fps = frame_count / (time.time() - start)
            draw_panel(frame, users[current], current, NUM_USERS, fps)

            out_frame = fit_to_display(frame, DISPLAY_W, DISPLAY_H)
            out = displayport.newframe()
            out[:] = out_frame
            displayport.writeframe(out)

    except KeyboardInterrupt:
        print(f"\nStopped. Avg FPS: {fps:.1f}")
        for i, u in enumerate(users):
            print(f"  {u.name}: squats={u.squat.count} "
                  f"pushups={u.pushup.count} situps={u.situp.count}")
    finally:
        cap.release()
        displayport.close()


if __name__ == "__main__":
    main()